# Experiment: `shorter.mp4` — Model Comparison with Correctness Scoring

Compare transcription quality across ASR approaches on a single 11-min Bulgarian video.

**File:** `shorter.mp4` — 11.2 min, 720p, 48 kbps mono audio

**Models tested (3 finalists):**
| # | Model | Type | Score |
|---|---|---|---|
| 1 | Groq API (`large-v3`) | Cloud API (free) | **98.8%** |
| 2 | `large-v3` (local) | Local Whisper 1.5B | 98.6% |
| 3 | `medium` (local) | Local Whisper 769M | 98.3% |

**Models removed (poor quality):**
| Model | Issue |
|---|---|
| `small` | Complete failure — all `...` output |
| `turbo` | Heavy hallucinations on low-bitrate BG |
| `sam8000/bg-finetuned` | Collapsed 11 min into 8 segments |
| `wav2vec2-bg` (infinitejoy) | Garbled, no word boundaries |

## 1. Setup

In [ ]:
import json
import os
import re
import subprocess
from pathlib import Path
from collections import Counter

# Fix ffmpeg
try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
except (subprocess.CalledProcessError, OSError):
    os.environ["PATH"] = "/usr/bin:" + os.environ.get("PATH", "")

PROJECT_ROOT = Path(".").resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "output"

# Groq key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

model_files = {
    "medium": "shorter_medium.txt",
    "large-v3": "shorter_large-v3.txt",
    "groq": "shorter_groq_large_v3.txt",
}

for model_name, filename in model_files.items():
    fpath = OUTPUT_DIR / filename
    if fpath.exists():
        content = fpath.read_text(encoding="utf-8")
        lines = [l.strip() for l in content.strip().split("\n") if l.strip()]
        transcripts[model_name] = {
            "lines": lines,
            "text": " ".join(re.sub(r"^\[.*?\]\s*", "", l) for l in lines),
            "n_segments": len(lines),
        }
        print(f"{model_name:15s} -> {len(lines):3d} segments, {len(transcripts[model_name]['text']):5d} chars")
    else:
        print(f"{model_name:15s} -> NOT FOUND ({fpath.name})")

print(f"\nLoaded {len(transcripts)} transcripts.")

In [ ]:
# Load all shorter_*.txt results
transcripts = {}

model_files = {
    "medium": "shorter_medium.txt",
    "large-v3": "shorter_large-v3.txt",
    "groq": "shorter_groq_large_v3.txt",
    "bg_finetuned": "shorter_bg_finetuned.txt",
}

for model_name, filename in model_files.items():
    fpath = OUTPUT_DIR / filename
    if fpath.exists():
        content = fpath.read_text(encoding="utf-8")
        lines = [l.strip() for l in content.strip().split("\n") if l.strip()]
        transcripts[model_name] = {
            "lines": lines,
            "text": " ".join(re.sub(r"^\[.*?\]\s*", "", l) for l in lines),
            "n_segments": len(lines),
        }
        print(f"{model_name:15s} -> {len(lines):3d} segments, {len(transcripts[model_name]['text']):5d} chars")
    else:
        print(f"{model_name:15s} -> NOT FOUND ({fpath.name})")

print(f"\nLoaded {len(transcripts)} transcripts.")

## 3. Correctness Scoring

Since we don't have a ground-truth transcript, we use **proxy metrics** to estimate quality:

| Metric | Weight | What it measures |
|---|---|---|
| **Bulgarian char ratio** | 25% | % of Cyrillic characters (higher = less hallucination) |
| **Non-hallucination score** | 25% | Absence of non-Bulgarian/non-punctuation characters |
| **Content density** | 15% | Meaningful text vs silence markers (`...`) |
| **Segment consistency** | 15% | Reasonable segment count (not collapsed or over-split) |
| **Repetition penalty** | 10% | Lower score for repeated phrases (hallucination pattern) |
| **Timestamp coverage** | 10% | How much of the 11 min audio is covered |

In [ ]:
AUDIO_DURATION = 674.0  # seconds

def score_transcript(name: str, data: dict) -> dict:
    """Score a transcript on multiple quality dimensions."""
    text = data["text"]
    lines = data["lines"]
    n_segments = data["n_segments"]
    
    # --- 1. Bulgarian character ratio (25%) ---
    # Cyrillic chars vs total alphabetic chars
    cyrillic = len(re.findall(r'[\u0400-\u04FF]', text))
    latin = len(re.findall(r'[a-zA-Z]', text))
    total_alpha = cyrillic + latin
    bg_ratio = cyrillic / max(total_alpha, 1)
    bg_score = min(bg_ratio / 0.95, 1.0)  # 95%+ Cyrillic = perfect
    
    # --- 2. Non-hallucination score (25%) ---
    # Penalize non-Cyrillic, non-punctuation, non-digit characters
    allowed = len(re.findall(r'[\u0400-\u04FF\s\d.,!?;:\-…\'\"()\[\]]', text))
    total_chars = max(len(text), 1)
    clean_ratio = allowed / total_chars
    halluc_score = min(clean_ratio / 0.95, 1.0)
    
    # --- 3. Content density (15%) ---
    # Penalize empty segments, '...' markers, very short segments
    content_lines = [l for l in lines if not re.match(r'^\[.*\]\s*[\.…\s]*$', l)]
    density = len(content_lines) / max(n_segments, 1)
    density_score = min(density / 0.80, 1.0)  # 80%+ content lines = perfect
    
    # --- 4. Segment consistency (15%) ---
    # Expect ~1 segment per 5-10 seconds for 674s audio = ~67-135 segments
    expected_min, expected_max = 40, 200
    if expected_min <= n_segments <= expected_max:
        segment_score = 1.0
    elif n_segments < expected_min:
        segment_score = max(n_segments / expected_min, 0.1)
    else:
        segment_score = max(1.0 - (n_segments - expected_max) / expected_max, 0.1)
    
    # --- 5. Repetition penalty (10%) ---
    # Detect repeated phrases (a hallucination pattern)
    words = text.split()
    if len(words) > 10:
        # Check for 3-gram repetitions
        trigrams = [" ".join(words[i:i+3]) for i in range(len(words)-2)]
        trigram_counts = Counter(trigrams)
        repeated = sum(c - 1 for c in trigram_counts.values() if c > 2)
        repeat_ratio = repeated / max(len(trigrams), 1)
        repetition_score = max(1.0 - repeat_ratio * 5, 0.0)
    else:
        repetition_score = 0.5
    
    # --- 6. Timestamp coverage (10%) ---
    # Parse timestamps and check coverage
    max_time = 0
    for line in lines:
        m = re.search(r'(\d+):(\d+\.\d+)\s*\]', line)
        if m:
            t = int(m.group(1)) * 60 + float(m.group(2))
            max_time = max(max_time, t)
    coverage = min(max_time / AUDIO_DURATION, 1.0) if max_time > 0 else 0.0
    coverage_score = coverage
    
    # --- Weighted total ---
    total = (
        bg_score * 0.25 +
        halluc_score * 0.25 +
        density_score * 0.15 +
        segment_score * 0.15 +
        repetition_score * 0.10 +
        coverage_score * 0.10
    )
    
    return {
        "model": name,
        "total_score": round(total * 100, 1),
        "bg_chars_%": round(bg_ratio * 100, 1),
        "halluc_score": round(halluc_score * 100, 1),
        "density_%": round(density * 100, 1),
        "segments": n_segments,
        "segment_score": round(segment_score * 100, 1),
        "repetition_score": round(repetition_score * 100, 1),
        "coverage_%": round(coverage * 100, 1),
        "text_length": len(text),
        "latin_chars": latin,
    }

In [ ]:
# Score all transcripts
scores = []
for name, data in transcripts.items():
    s = score_transcript(name, data)
    scores.append(s)

# Sort by total score
scores.sort(key=lambda x: x["total_score"], reverse=True)

# Display results
print(f"{'Rank':<5} {'Model':<16} {'TOTAL':>6} {'BG%':>6} {'Clean%':>7} {'Dense%':>7} {'Segs':>5} {'SegSc':>6} {'Rept%':>6} {'Cover%':>7} {'Chars':>6} {'Latin':>6}")
print("-" * 105)
for i, s in enumerate(scores, 1):
    print(f"{i:<5} {s['model']:<16} {s['total_score']:>5.1f}% {s['bg_chars_%']:>5.1f}% {s['halluc_score']:>6.1f}% {s['density_%']:>6.1f}% {s['segments']:>5} {s['segment_score']:>5.1f}% {s['repetition_score']:>5.1f}% {s['coverage_%']:>6.1f}% {s['text_length']:>6} {s['latin_chars']:>5}")

## 4. Visual Comparison

In [ ]:
# Bar chart of scores
print("\n=== CORRECTNESS SCORE ===")
print()
for s in scores:
    bar_len = int(s["total_score"] / 2)
    bar = "\u2588" * bar_len + "\u2591" * (50 - bar_len)
    star = " *** BEST" if s == scores[0] else ""
    print(f"  {s['model']:<16} {bar} {s['total_score']:>5.1f}%{star}")

print()
print("=== BREAKDOWN ===")
print()
metrics = [
    ("BG chars %", "bg_chars_%", "Higher = more Bulgarian, less hallucination"),
    ("Clean text %", "halluc_score", "Higher = fewer foreign/garbage characters"),
    ("Content density", "density_%", "Higher = fewer empty/silence segments"),
    ("Repetition", "repetition_score", "Higher = fewer repeated phrases"),
    ("Coverage", "coverage_%", "Higher = more of the 11 min audio transcribed"),
]

for metric_name, key, desc in metrics:
    print(f"  {metric_name} ({desc}):")
    for s in scores:
        val = s[key]
        bar = "\u2588" * int(val / 5) + "\u2591" * (20 - int(val / 5))
        print(f"    {s['model']:<16} {bar} {val:>5.1f}%")
    print()

## 5. Side-by-Side Text Comparison

Compare the same time ranges across models.

In [ ]:
def get_text_at_time(lines, target_start, window=10):
    """Get transcript text near a given timestamp."""
    results = []
    for line in lines:
        m = re.match(r'\[(\d+):(\d+\.\d+)', line)
        if m:
            t = int(m.group(1)) * 60 + float(m.group(2))
            if abs(t - target_start) < window:
                text = re.sub(r'^\[.*?\]\s*', '', line)
                if text.strip() and text.strip() != '...':
                    results.append(text.strip())
    return " | ".join(results[:3]) if results else "(no text)"

# Compare at key timestamps
checkpoints = [0, 30, 60, 120, 240, 360, 480, 540, 600]

print(f"{'Time':>6s}", end="")
for name in transcripts:
    print(f"  {name:<25s}", end="")
print()
print("-" * (6 + 27 * len(transcripts)))

for t in checkpoints:
    m, s = divmod(t, 60)
    print(f"{int(m):02d}:{int(s):02d} ", end="")
    for name, data in transcripts.items():
        snippet = get_text_at_time(data["lines"], t, window=15)
        print(f"  {snippet[:25]:<25s}", end="")
    print()

## 6. Summary & Recommendation

In [ ]:
best = scores[0]
print(f"BEST MODEL: {best['model']}")
print(f"  Total Score:    {best['total_score']}%")
print(f"  BG Character %: {best['bg_chars_%']}%")
print(f"  Clean Text %:   {best['halluc_score']}%")
print(f"  Segments:       {best['segments']}")
print(f"  Latin chars:    {best['latin_chars']} (fewer = better)")
print()
print("RANKING:")
for i, s in enumerate(scores, 1):
    print(f"  {i}. {s['model']:<16} {s['total_score']:>5.1f}%")